VASP analysis by Peter H. Jacobse 2025

In [2]:
import os
import numpy as np
from numpy import sin, cos, sqrt, pi, exp, mod
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection, LineCollection
import open3d as o3d
from pymatgen.core import Lattice, Structure
from pymatgen.electronic_structure.core import Spin
import pymatgen.io.vasp.outputs as vasp_out
from scipy.ndimage import zoom
from scipy.signal import convolve2d
import h5py as h5
from pymatgen.io.ase import AseAtomsAdaptor
from sympy import Matrix, symbols, pprint
#from unfold import make_kpath, removeDuplicateKpoints, find_K_from_k

dark_blue = (0, 0, .7, 1)
dark_blue_transparent = (0, 0, .7, .2)
dark_blue_semitransparent = (0, 0, .7, .5)
dark_red = (.8, 0, 0, 1)
dark_red_transparent = (.8, 0, 0, .2)
dark_red_semitransparent = (.8, 0, 0, .5)
dark_green = (0, .4, 0, 1)
dark_green_transparent = (0, .4, 0, .2)
dark_green_semitransparent = (0, .4, 0, .5)
dark_purple = (.7, 0, .4, 1)
dark_purple_transparent = (.8, 0, .7, .2)
dark_purple_semitransparent = (.8, 0, .7, .5)
dark_cyan = (.4, .1, .7, 1)

def colorfunction_hue(phase: float = 0, contrast: float = 1, brightness: float = 1): return(.5 * brightness + .499 * np.cos(phase) * contrast, .5 * brightness + .499 * np.cos(phase + 2 * np.pi / 3) * contrast, .5 * brightness + .499 * np.cos(phase + 4 * np.pi / 3) * contrast, 1)

def calculate_electrons(structure):
    n_elec = sum([1 * int(str(structure.species[i]) == "H") for i in range(len(structure.sites))]) + sum([3 * int(str(structure.species[i]) == "B") for i in range(len(structure.sites))]) + sum([4 * int(str(structure.species[i]) == "C") for i in range(len(structure.sites))]) + sum([5 * int(str(structure.species[i]) == "N") for i in range(len(structure.sites))])
    return n_elec

def get_color_and_radius(specie, opacity: float = 1, contrast: float = 1, brightness: float = 1, emphasize_heteroatoms: bool = False):
    """Return the color and van der Waals radius for a given atomic specie."""
    match str(specie):
        case "W":
            vdW_radius = 2.49
            atom_color = [.6, .2, .6]
        case "Cr":
            vdW_radius = 1.39
            atom_color = [.5, .5, .5]
        case "Si":
            vdW_radius = 1.2
            atom_color = [.5, .5, .5]
        case "C":
            vdW_radius = 1.7
            atom_color = [.1, .1, .1]
        case "O":
            vdW_radius = 1.52
            if emphasize_heteroatoms: vdW_radius *= 1.5
            atom_color = [1, 0, 0]
        case "H":
            vdW_radius = 1.2
            atom_color = [.9, .9, .9]
        case "N":
            vdW_radius = 1.55
            if emphasize_heteroatoms: vdW_radius *= 1.5
            atom_color = [0, 0, 1]
        case "P":
            vdW_radius = 1.8
            if emphasize_heteroatoms: vdW_radius *= 1.5
            atom_color = [1, 0.5, 0]
        case "B":
            vdW_radius = 1.92
            if emphasize_heteroatoms: vdW_radius *= 1.5
            atom_color = [.8, .5, .5]
        case "Fe":
            vdW_radius = 1.32
            atom_color = [.5, 0, 0]
        case "S":
            vdW_radius = 1.8
            atom_color = [.9, .9, 0]
        case _:
            vdW_radius = 1.5
            atom_color = [.5, .5, .5]
    
    if 0 < contrast < 1: atom_color = contrast * np.array(atom_color) + np.array([.5, .5, .5]) * (1 - contrast)
    atom_color += (np.ones_like(atom_color) * brightness - .5)
    atom_color = np.clip(atom_color, a_min = 0, a_max = 1)

    atom_color = np.append(atom_color, opacity)
    return atom_color, vdW_radius

def build_GNR(N: int = 2, orientation: str = "A"):
    xlist = np.zeros(N * 2, dtype = float)
    ylist = np.zeros(N * 2, dtype = float)
    atomlist = np.zeros(N * 2, dtype = str)
    CC_length = 1.42
    CH_length = 1.09
    if orientation == "A":
        row_to_row = .25 * CC_length * sqrt(3) # Row-to-row spacing in the y direction
        for i in range(N):
            xdist_from_axis = .5 * CC_length * (np.mod(i, 2) + 1)
            sign = 1 - 2 * mod(i, 2)
            xlist[i] = sign * xdist_from_axis
            xlist[i + N] = -sign * xdist_from_axis
            ylist[i] = 2 * i * row_to_row
            ylist[i + N] = 2 * i * row_to_row
            atomlist[i] = "C"
            atomlist[i + N] = "C"

            # Pad hydrogen atoms to the bottom and top of the GNR
        xlist = np.append(xlist, [.5 * CC_length + .5 * CH_length, -.5 * CC_length - .5 * CH_length,
                                -CC_length * (.25 * mod((N + 1) * 2, 4) + .5) + CH_length * (.5 - np.mod(N, 2)), CC_length * (.25 * mod((N + 1) * 2, 4) + .5) - CH_length * (.5 - mod(N, 2))])
        ylist = np.append(ylist, [-CH_length * sqrt(3) / 2, -CH_length * sqrt(3) / 2,
                                max(ylist) + CH_length * sqrt(3) / 2, max(ylist) + CH_length * sqrt(3) / 2])
        atomlist = np.append(atomlist, ["H", "H", "H", "H"])
        latvec_x = 3 * CC_length # Simply the unit cell length
        latvec_y = (N + 1) * 4 * row_to_row # The GNR hard wall boundary conditions dictate that the transverse component of the wavefunctions have nodal planes on atomic row 0 and atomic row N + 1
        # Thus, using this width ensures that the unit cell exactly fits the wavelength of the transverse waves and integer multiples of it
        # This prevents having to represent the transverse nodal plane structure with lots of different waves, also known as Fourier leakage
    else:
        row_to_row = 1.5 * CC_length
        for i in range(0, N * 2, 2):
            xlist[i] = -.25 * sqrt(3) * CC_length * (mod(i, 4) - 1) # Atom to the left side of the x axis
            xlist[i + 1] = .25 * sqrt(3) * CC_length * (mod(i, 4) - 1) # Atom to the right side of the x axis
            ylist[i] = .5 * i * row_to_row - .25 * CC_length
            ylist[i + 1] = .5 * i * row_to_row + .25 * CC_length
            atomlist[i] = "C"
            atomlist[i + 1] = "C"
        xlist = np.append(xlist, [xlist[0], xlist[-1]])
        ylist = np.append(ylist, [-.25 * CC_length - CH_length, ylist[-1] + CH_length])
        atomlist = np.append(atomlist, ["H", "H"])
        latvec_x = sqrt(3) * CC_length # Simply the unit cell length
        latvec_y = (N + 1) * 2 * row_to_row # The GNR hard wall boundary conditions dictate that the transverse component of the wavefunctions have nodal planes on atomic row 0 and atomic row N + 1
        # Thus, using this width ensures that the unit cell exactly fits the wavelength of the transverse waves and integer multiples of it
        # This prevents having to represent the transverse nodal plane structure with lots of different waves, also known as Fourier leakage

    ylist -= np.mean(ylist) # Shift the y coordinates so that the structure is centered around the origin

    latvec_z = 10 # 1 nm distance in the height direction should be sufficient to limit interactions between periodic images

    xlist_shifted = xlist + .5 * latvec_x # Move the atoms from being centered around the origin to being centered around the center of the unit cell
    ylist_shifted = ylist + .5 * latvec_y

    structure = Structure(lattice = Lattice.from_parameters(latvec_x, latvec_y, latvec_z, 90, 90, 90), species = atomlist,
                        coords = np.array([xlist_shifted, ylist_shifted, np.zeros_like(xlist) + .5 * latvec_z]).T, coords_are_cartesian = True)
    return structure

def find_neighbors(structure, min_bondlength: float = 1., max_bondlength: float = 1.6, exclude_hydrogen: bool = False):
    xyz_list = []

    for i in range(len(structure.species)):
        element_i = structure.sites[i].species_string
        if element_i != "H":
            xyz_list.append(structure.sites[i].coords)
        if element_i == "H" and not exclude_hydrogen:
            xyz_list.append(structure.sites[i].coords)
    xyz_list = np.array(xyz_list)
    
    n_atoms = len(xyz_list)
    min_bondlength2 = min_bondlength ** 2
    max_bondlength2 = max_bondlength ** 2
    connectivity_matrix = np.zeros((n_atoms, n_atoms), dtype = int)
    connectivity_matrix_x = np.zeros_like(connectivity_matrix)
    
    latvec_x = structure.lattice.a
    for i in range(n_atoms): # Loop over all the atoms
        for j in range(n_atoms): # Loop over atom pairs for bond detection
            vec_ij = xyz_list[j] - xyz_list[i] # Add the lattice vector in x to calculate neighbors with the nearest cells
            dist2_ij = sum(vec_ij ** 2) # Detect the bond distance
            if dist2_ij < max_bondlength2 and dist2_ij > min_bondlength2: # Create a bond when the atoms are closer than the max bond distance
                connectivity_matrix[i, j] = 1
            
            vec_ij += [latvec_x, 0, 0] # Now check bonds between the periodic image shifted in the direction of the first lattice vector
            dist2_ij = sum(vec_ij ** 2) # Detect the bond distance
            if dist2_ij < max_bondlength2 and dist2_ij > min_bondlength2: # Create a bond when the atoms are closer than the max bond distance
                connectivity_matrix_x[i, j] = 1
    
    #connectivity_matrix_ext = connectivity_matrix + connectivity_matrix_x + connectivity_matrix_x.T # This matrix contains the internal bonds plus the bonds protruding the unit cell boundary
    angles_matrix = np.linalg.matrix_power(connectivity_matrix, 2) # This matrix contains all paths of length 2
    #angles_matrix_ext = np.linalg.matrix_power(connectivity_matrix_ext, 2) # This matrix contains all paths of length 2, including to atoms in the neighboring unit cell
    np.fill_diagonal(angles_matrix, 0) # Remove paths that connect atoms to themselves
    #np.fill_diagonal(angles_matrix_ext, 0) # Remove paths that connect atoms to themselves
    #angles_matrix_x = angles_matrix_ext
    #angles_matrix_x[angles_matrix_ext != 0] = 0 # Remove elements that are already in the intra-cell matrix
    
    #dihedrals_matrix = np.linalg.matrix_power(connectivity_matrix, 3) # This matrix has paths of length 3, i.e. third nearest neighbors
    #dihedrals_matrix_ext = np.linalg.matrix_power(connectivity_matrix_ext, 3)
    #dihedrals_matrix[connectivity_matrix != 0] = 0 # Remove paths that connect atoms to nearest neighbors (paths that go back and forth along one bond once)
    #dihedrals_matrix_ext[connectivity_matrix_ext != 0] = 0
    
    return connectivity_matrix

def extend_structure(structure, supercell: tuple = (1, 1, 1), max_bondlength = 1.8):
    superstructure = structure.copy() # Create a superstructure, using the user-defined supercell parameters
    if supercell != (1, 1, 1) and type(supercell) == tuple and len(supercell) == 3: superstructure.make_supercell(supercell)
    superlattice = superstructure.lattice
    latvec_x = superlattice.a
    latvec_y = superlattice.b

    structure_ext = superstructure.copy() # Extend the structure beyond the unit cell to account for bonds that protrude through the unit cell boundaries
    structure_ext.make_supercell((3, 1, 1))
    structure_ext.translate_sites(vector = [-latvec_x, 0, 0], indices = range(len(structure_ext.sites)), frac_coords = False, to_unit_cell = False)
    remove_list = [] # Remove the atoms beyond the unit cell boundaries that do not create bonds
    for i in range(len(structure_ext.sites)):
        if not -max_bondlength < structure_ext.sites[i].coords[0] < latvec_x + max_bondlength: remove_list.append(i)
    structure_ext.remove_sites(remove_list) # These are the atoms that will be plotted

    return latvec_x, latvec_y, structure_ext

def create_2D_model(structure, atom_size: float = 0.25, bond_size: float = 0.9, max_bondlength: float = 1.8, min_bondlength: float = 1., max_bondlength2: float = 2.6, max_bondlength3: float = 2.9, opacity: float = 1, contrast: float = 1, brightness: float = .5, model_type: str = "standard", emphasize_heteroatoms: bool = False):
    site_list = structure.sites # These are the atoms that will be plotted

    atoms = []
    bonds = []
    
    if model_type == "TB":
        bond_matrix = find_neighbors(structure, max_bondlength = max_bondlength, min_bondlength = min_bondlength, exclude_hydrogen = True)
        bond_matrix_2NN = find_neighbors(structure, max_bondlength = max_bondlength2, min_bondlength = max_bondlength, exclude_hydrogen = True)
        bond_matrix_3NN = find_neighbors(structure, max_bondlength = max_bondlength3, min_bondlength = max_bondlength2, exclude_hydrogen = True)
    else:
        bond_matrix = find_neighbors(structure, max_bondlength = max_bondlength, min_bondlength = min_bondlength)
        bond_matrix_2NN = bond_matrix_3NN = np.zeros_like(bond_matrix)
    n_atoms = len(bond_matrix)

    for atom_i in range(n_atoms): # Loop over all the atoms
        site_i = site_list[atom_i] # Extract information
        specie_i = site_i.specie
        xyz_i = site_i.coords
        atom_color_i, vdW_radius_i = get_color_and_radius(specie_i, opacity, contrast, brightness, emphasize_heteroatoms)
        radius_i = atom_size * vdW_radius_i
        atom = plt.Circle((xyz_i[0], xyz_i[1]), radius_i, color = atom_color_i)
        atoms.append(atom)
        
        for atom_j in range(atom_i + 1, n_atoms): # Loop over atom pairs for bond detection
            if bool(bond_matrix[atom_i, atom_j]):
                site_j = site_list[atom_j]
                specie_j = site_j.specie
                xyz_j = site_j.coords
                vec_ij = xyz_j - xyz_i
                dist_ij = np.linalg.norm(vec_ij) # Detect the bond distance

                atom_color_j, vdW_radius_j = get_color_and_radius(specie_j, opacity, contrast, brightness, emphasize_heteroatoms)
                normvec_ij = vec_ij / dist_ij # Calculate the normalized vector in order to determine the orientation of the bond in 3D space
                if abs(normvec_ij[0]) < .000001: angle_ij = .5 * np.pi * np.sign(normvec_ij[1]) # Avoid division by zero errors
                else: angle_ij = np.arctan(normvec_ij[1] / normvec_ij[0])
                
                xyz_avg = .5 * xyz_i + .5 * xyz_j
                bond_color = .5 * np.array(atom_color_i) + .5 * np.array(atom_color_j) # Set the bond color to be the average of the colors associated with the two atoms
                if str(specie_i) == "H" or str(specie_j) == "H": # If one of the atoms is a hydrogen atom, use a white bond instead
                    bond_color = np.array([0.8, 0.8, 0.8, opacity])
                if model_type == "TB": bond_color = np.array([.1, .1, .1, opacity])
                height = bond_size * atom_size * 1.2
                bond = plt.Rectangle((xyz_avg[0] - .5 * dist_ij, xyz_avg[1] - .5 * height), dist_ij, height, angle = 180 * angle_ij / np.pi, rotation_point = "center", color = bond_color)
                bonds.append(bond)
            if model_type == "TB":
                if bool(bond_matrix_3NN[atom_i, atom_j]):
                    site_j = site_list[atom_j]
                    specie_j = site_j.specie
                    xyz_j = site_j.coords
                    vec_ij = xyz_j - xyz_i
                    dist_ij = np.linalg.norm(vec_ij) # Detect the bond distance

                    atom_color_j, vdW_radius_j = get_color_and_radius(specie_j, opacity, contrast, brightness)
                    normvec_ij = vec_ij / dist_ij # Calculate the normalized vector in order to determine the orientation of the bond in 3D space
                    if abs(normvec_ij[0]) < .000001: angle_ij = .5 * np.pi * np.sign(normvec_ij[1]) # Avoid division by zero errors
                    else: angle_ij = np.arctan(normvec_ij[1] / normvec_ij[0])
                    
                    xyz_avg = .5 * xyz_i + .5 * xyz_j
                    bond_color = np.array([0, .5, .8, opacity])
                    height = bond_size * atom_size * 1.2
                    bond = plt.Rectangle((xyz_avg[0] - .5 * dist_ij, xyz_avg[1] - .5 * height), dist_ij, height, angle = 180 * angle_ij / np.pi, rotation_point = "center", color = bond_color)
                    bonds.append(bond)
                if bool(bond_matrix_2NN[atom_i, atom_j]):
                    site_j = site_list[atom_j]
                    specie_j = site_j.specie
                    xyz_j = site_j.coords
                    vec_ij = xyz_j - xyz_i
                    dist_ij = np.linalg.norm(vec_ij) # Detect the bond distance

                    atom_color_j, vdW_radius_j = get_color_and_radius(specie_j, opacity, contrast, brightness)
                    normvec_ij = vec_ij / dist_ij # Calculate the normalized vector in order to determine the orientation of the bond in 3D space
                    if abs(normvec_ij[0]) < .000001: angle_ij = .5 * np.pi * np.sign(normvec_ij[1]) # Avoid division by zero errors
                    else: angle_ij = np.arctan(normvec_ij[1] / normvec_ij[0])
                    
                    xyz_avg = .5 * xyz_i + .5 * xyz_j
                    bond_color = np.array([.8, .5, 0, opacity])
                    height = bond_size * atom_size * 1.2
                    bond = plt.Rectangle((xyz_avg[0] - .5 * dist_ij, xyz_avg[1] - .5 * height), dist_ij, height, angle = 180 * angle_ij / np.pi, rotation_point = "center", color = bond_color)
                    bonds.append(bond)
    return atoms, bonds

def create_3D_model(structure, atom_size: float = 0.25, bond_size: float = 0.9, max_bondlength: float = 1.8, min_bondlength: float = 1., resolution: int = 20, opacity: float = 1, contrast: float = 1, brightness: float = .5, emphasize_heteroatoms: bool = False):
    site_list = structure.sites
        
    atoms = []
    bonds = []
    
    bond_matrix = find_neighbors(structure, max_bondlength = max_bondlength, min_bondlength = min_bondlength, exclude_hydrogen = False)
    n_atoms = len(bond_matrix)

    for atom_i in range(n_atoms): # Loop over all the atoms
        site_i = site_list[atom_i] # Extract information
        specie_i = site_i.specie
        xyz_i = site_i.coords
        atom_color_i, vdW_radius_i = get_color_and_radius(specie_i, opacity, contrast, brightness, emphasize_heteroatoms)
        radius_i = atom_size * vdW_radius_i

        # Create a sphere for the current atom
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius = radius_i, resolution = resolution)
        sphere.paint_uniform_color(atom_color_i[:3])
        sphere.translate((xyz_i[0], xyz_i[1], xyz_i[2]))
        sphere.compute_vertex_normals()
        atoms += [sphere]

        for atom_j in range(atom_i + 1, n_atoms): # Loop over atom pairs for bond detection
            if bool(bond_matrix[atom_i, atom_j]):
                site_j = site_list[atom_j]
                specie_j = site_j.specie
                xyz_j = site_j.coords
                vec_ij = xyz_j - xyz_i
                dist_ij = np.linalg.norm(vec_ij) # Detect the bond distance
            
                atom_color_j, vdW_radius_j = get_color_and_radius(specie_j, opacity, contrast, brightness, emphasize_heteroatoms)
                normvec_ij = vec_ij / dist_ij # Calculate the normalized vector in order to determine the orientation of the bond in 3D space
                bond_color = 0.5 * np.array(atom_color_i) + 0.5 * np.array(atom_color_j) # Set the bond color to be the average of the colors associated with the two atoms
                if str(specie_i) == "H" or str(specie_j) == "H": # If one of the atoms is a hydrogen atom, use a white bond instead
                    bond_color = np.array([0.8, 0.8, 0.8])

                # Create a cylinder for the bond
                z_axis = np.array([0, 0, 1])
                rotation_axis = np.cross(z_axis, normvec_ij) # Calculate the rotation matrix from the angle between the bond unit vector and the z axis
                if np.linalg.norm(rotation_axis) < 1e-8:
                    Rot_matrix = np.eye(3)
                else:
                    rotation_axis /= np.linalg.norm(rotation_axis)
                    rotation_angle = np.arccos(np.clip(np.dot(z_axis, normvec_ij), -1.0, 1.0))
                    Rot_matrix = o3d.geometry.get_rotation_matrix_from_axis_angle(rotation_axis * rotation_angle)
            
                cylinder = o3d.geometry.TriangleMesh.create_cylinder(radius = bond_size * atom_size * 1.2, height = dist_ij, resolution = resolution, split = 4)
                cylinder.paint_uniform_color(bond_color[:3])
                cylinder.rotate(Rot_matrix, center = (0, 0, 0))
                cylinder.translate((xyz_i + xyz_j) / 2)
                cylinder.compute_vertex_normals()
                bonds += [cylinder]
    
    return atoms, bonds

def structure_plot(structure, supercell: tuple = (1, 1, 1), atom_size: float = 0.25, bond_size: float = 0.9, max_bondlength: float = 1.6, resolution: int = 20, dimensions: int = 2, opacity: float = 1, contrast: float = 1, brightness: float = .5, return_type: str = "show", model_type: str = "standard", emphasize_heteroatoms: bool = False, width: int = 600, height: int = 400):
    latvec_x, latvec_y, structure_ext = extend_structure(structure, supercell = supercell, max_bondlength = max_bondlength)
    latvec_z = structure_ext.lattice.matrix[2]
    supercell_center = np.array([.5 * latvec_x, .5 * latvec_y, 0]) + .5 * latvec_z

    if dimensions == 2:
        atoms, bonds = create_2D_model(structure_ext, atom_size = atom_size, bond_size = bond_size, max_bondlength = max_bondlength, opacity = opacity, contrast = contrast, brightness = brightness, model_type = model_type, emphasize_heteroatoms = emphasize_heteroatoms)
        bonds_collection = PatchCollection(bonds, match_original = True)
        atoms_collection = PatchCollection(atoms, match_original = True)
        if return_type == "collections": return atoms_collection, bonds_collection # Return the bare patch collections so the user can make their own plot
        else:
            fig, ax = plt.subplots() # Create the plot
            plt.tight_layout()
            ax.set_aspect(1)
            ax.set_xbound([0, latvec_x])
            ax.set_ybound([0, latvec_y])
            ax.add_collection(bonds_collection)
            ax.add_collection(atoms_collection)
            if return_type == "figure": return fig # Return the fig object
            else: plt.show() # Simply show the figure
    
    elif dimensions == 3:
        atoms, bonds = create_3D_model(structure_ext, atom_size = atom_size, bond_size = bond_size, max_bondlength = max_bondlength, resolution = resolution, opacity = opacity, contrast = contrast, brightness = brightness, emphasize_heteroatoms = emphasize_heteroatoms)
        if return_type == "collections": return atoms, bonds # Return the bare patch collections so the user can make their own plot
        else: o3d.visualization.draw_geometries(bonds + atoms, lookat = supercell_center, zoom = 0.3, up = [0, 1, 0], width = width, height = height)

    else: print("Unknown dimensionality")

def tb_orbital_plot(structure, supercell: tuple = (1, 1, 1), atom_size: float = 0.25, bond_size: float = 0.9, max_bondlength: float = 1.6, opacity: float = 1, contrast: float = 1, brightness: float = .5, return_type: str = "show", model_type: str = "standard", emphasize_heteroatoms: bool = False, width: int = 600, height: int = 400):
    """ Plot the structure as svg, so the orbitals can be overlaid """
    latvec_x, latvec_y, structure_ext = extend_structure(structure, supercell = supercell, max_bondlength = max_bondlength)

    # To do: add disks containing the wfn information

    atoms, bonds = create_2D_model(structure_ext, atom_size = atom_size, bond_size = bond_size, max_bondlength = max_bondlength, opacity = opacity, contrast = contrast, brightness = brightness, model_type = model_type, emphasize_heteroatoms = emphasize_heteroatoms)
    bonds_collection = PatchCollection(bonds, match_original = True)
    atoms_collection = PatchCollection(atoms, match_original = True)
    if return_type == "collections": return atoms_collection, bonds_collection
    else:
        fig, ax = plt.subplots() # Create the plot
        plt.tight_layout()
        ax.set_aspect(1)
        ax.set_xbound([0, latvec_x])
        ax.set_ybound([0, latvec_y])
        ax.add_collection(bonds_collection)
        ax.add_collection(atoms_collection)
        if return_type == "figure": return fig
        else: plt.show()

def procar_loader(folder_name: str, verbose: bool = True):
    procar_data = vasp_out.Procar(folder_name + "PROCAR")
    structure_file_name = folder_name + "CONTCAR"
    structure, lat_vec, center = structure_loader(structure_file_name, show = False)

    n_bands = procar_data.nbands
    n_kpoints = procar_data.nkpoints
    n_ions = procar_data.nions
    n_noH = sum([int(str(structure.sites[i].specie) != "H") for i in range(len(structure.sites))]) # The number of atoms that are not hydrogen atoms
    spin_polarized = bool(procar_data.nspins - 1)
    
    orbital_list = procar_data.orbitals
    pz_index = int(np.where(np.asarray(orbital_list) == "pz")[0][0])
    n_orbitals = len(orbital_list)
    if verbose:
        print("Number of k points: " + str(n_kpoints) + "; number of bands: " + str(n_bands) + "; number of atoms: " + str(n_ions) + "; number of hydrogen atoms: " + str(n_ions - n_noH) + "; number of orbitals: " + str(n_orbitals))
        print("Identity of orbitals: ", orbital_list)

    CB_min_list = np.zeros(n_kpoints, dtype = float)
    VB_max_list = np.zeros(n_kpoints, dtype = float)

    for k_point in range(n_kpoints):
        luco_index = int(np.where(procar_data.occupancies[Spin.up][k_point] < .5)[0][0]) # Find the index of the eigenstate where the occupancy goes to zero
        CB_min_list[k_point] = procar_data.eigenvalues[Spin.up][k_point][luco_index]
        VB_max_list[k_point] = procar_data.eigenvalues[Spin.up][k_point][luco_index - 1]

    luco_energy = np.min(CB_min_list)
    hoco_energy = np.max(VB_max_list)
    midgap_energy = .5 * hoco_energy + .5 * luco_energy
    band_gap = luco_energy - hoco_energy

    if verbose:
        print("The VB maximum is at " + str(hoco_energy) + " eV and the CB minimum is at " + str(luco_energy) + " eV, giving a band gap of " + str(band_gap) + " eV")
        print("Spin polarized calculation: " + str(spin_polarized))

    return [n_bands, n_kpoints, n_ions, n_noH, n_orbitals, spin_polarized, orbital_list, pz_index, luco_index, hoco_energy, luco_energy, midgap_energy, band_gap, procar_data]

def wavecar_loader(folder_name: str, verbose: bool = True):
    if not os.path.exists(os.path.dirname(folder_name)):
        print("Folder not found!")
        return
    if not os.path.exists(os.path.dirname(folder_name) + "/WAVECAR"):
        print("WAVECAR file not found!")
        return
    if not os.path.exists(os.path.dirname(folder_name) + "/CONTCAR"):
        print("Associated CONTCAR file not found!")
    else: structure = Structure.from_file(os.path.dirname(folder_name) + "/CONTCAR")

    wavecar_data = vasp_out.Wavecar(os.path.dirname(folder_name) + "/WAVECAR") # Loading WAVECAR files might take a few seconds
    n_bands = wavecar_data.nb
    n_kpoints = wavecar_data.nk
    spin_polarized = bool(wavecar_data.spin - 1)

    if verbose: print("Number of k points: " + str(n_kpoints) + "; number of bands: " + str(n_bands))

    energies = wavecar_data.band_energy
    n2_kpoints = n_kpoints
    if spin_polarized: # Put the spin up data and spin down data behind each other in k-space for checking the band minima and maxima
        energies = np.vstack(energies)
        n2_kpoints = 2 * n_kpoints
    CB_min_list = np.zeros(n2_kpoints, dtype = float)
    VB_max_list = np.zeros(n2_kpoints, dtype = float)

    for k_point in range(n2_kpoints):
        luco_index = int(np.where(energies[k_point][:, 2] < .5)[0][0]) # Find the index of the eigenstate where the occupancy goes to zero
        CB_min_list[k_point] = energies[k_point][luco_index, 0]
        VB_max_list[k_point] = energies[k_point][luco_index - 1, 0]

    luco_energy = np.min(CB_min_list)
    hoco_energy = np.max(VB_max_list)
    midgap_energy = .5 * hoco_energy + .5 * luco_energy
    band_gap = luco_energy - hoco_energy

    if verbose:
        print("The VB maximum is at " + str(hoco_energy) + " eV and the CB minimum is at " + str(luco_energy) + " eV, giving a band gap of " + str(band_gap) + " eV")
        print("Spin polarized calculation: " + str(spin_polarized))
    
    return [n_bands, n_kpoints, spin_polarized, luco_index, hoco_energy, luco_energy, midgap_energy, band_gap, wavecar_data, structure]

def get_bands(wavecar_data, midgap_energy: float = 0):
    n_kpoints = wavecar_data.nk
    n_bands = wavecar_data.nb
    spin_polarized = bool(wavecar_data.spin - 1)

    sigma_bands0 = np.zeros((2, n_kpoints, n_bands, 5), dtype = float) # Initialize arrays for the sigma and pi bands
    pi_bands0 = np.zeros_like(sigma_bands0) # The arrays are initialized too large but will later be truncated to fit the number of bands

    for spin in range(2):
        if spin_polarized:
            all_coeffs = wavecar_data.coeffs[spin]
            all_eigs = wavecar_data.band_energy[spin]
        else:
            all_coeffs = wavecar_data.coeffs
            all_eigs =  wavecar_data.band_energy

        for k_point in range(n_kpoints):
            g_points = wavecar_data.Gpoints[k_point]
            coeffs_at_k = all_coeffs[k_point]
            eigs_at_k = all_eigs[k_point]
            
            n_sigma = 0 # The number of pi and sigma bands is counted for each k point
            n_pi = 0

            for band in range(n_bands):
                eigenenergy = eigs_at_k[band][0] - midgap_energy
                coeffs_band = coeffs_at_k[band] # Read the coefficients

                # Check if the wfn has more weight on nx = 0 (meaning it is symmetric wrt reflection in the yz plane) or on nx = 1 (antisymmetric wrt reflection in the yz plane)
                #g_points_list = [[0., 1., 1.], [1., 1., 1.], [2., 1., 1.], [3., 1., 1.]] # All wfns have weight on ny = 1 and nz = 1 due to the modulation incurred from the vacuum regions
                #indices = [(int(np.where(np.all(g_points == g_point, axis = 1))[0][0])) for g_point in g_points_list]
                #coeffs = np.abs(np.array([wave_band[index] for index in indices]))
                #nx = int(np.where(coeffs == np.max(coeffs))[0][0])
                #if coeffs[0] > coeffs[1]: nx = 0
                nx = 1

                # Check the wfn sigma/pi symmetry
                g_points_list = [[nx * 1., 1., 1.], [nx * 1., 1., -1.], [2., 2., 1.], [2., 2., -1.]] # Opposite k-points (reflected in z) are retrieved at nx = 0 and nx = 1
                indices = [(int(np.where(np.all(g_points == g_point, axis = 1))[0][0])) for g_point in g_points_list]
                coeffs = np.array([coeffs_band[index] for index in indices])
                if np.abs(np.diff(coeffs))[0] + np.abs(np.diff(coeffs))[2] < 10 ** (-5): symz = 0
                else: symz = 1
                nz = symz # If the wfn is a sigma orbital, focus only on wfns with nz = 0; otherwise, focus on wfns with nz = 1

                if nz < .5:
                    sigma_bands0[spin, k_point, n_sigma, 0] = eigenenergy
                    n_sigma += 1
                else: # For pi orbitals, determine more symmetries
                    pi_bands0[spin, k_point, n_pi, 0] = eigenenergy
                                        
                    g_points_list_ny = [[nx * 1., ny * 1., nz * 1.] for ny in range(8)]
                    g_points_list_minus_ny = [[nx * 1., -ny * 1., nz * 1.] for ny in range(8)]
                    indices_ny = [(int(np.where(np.all(g_points == g_point, axis = 1))[0][0])) for g_point in g_points_list_ny]
                    indices_minus_ny = [(int(np.where(np.all(g_points == g_point, axis = 1))[0][0])) for g_point in g_points_list_minus_ny]
                    coeffs_ny = np.array([coeffs_band[index] for index in indices_ny])
                    coeffs_minus_ny = np.array([coeffs_band[index] for index in indices_minus_ny])
                    sinlist = np.abs(np.array([coeffs_ny[ny] - coeffs_minus_ny[ny] for ny in range(8)]))
                    coslist = np.abs(np.array([coeffs_ny[ny] + coeffs_minus_ny[ny] for ny in range(8)]))
                    symy = np.sum(sinlist) / (np.sum(sinlist) + np.sum(coslist))
                    
                    pi_bands0[spin, k_point, n_pi, 1] = symy
                    n_pi += 1

    n_sigma_bands = np.min([np.array([np.where(np.diff(sigma_bands0[0, k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)]),
                            np.array([np.where(np.diff(sigma_bands0[1, k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)])]) # Determine where to crop the band arrays
    n_pi_bands = np.min([np.array([np.where(np.diff(pi_bands0[0, k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)]),
                        np.array([np.where(np.diff(pi_bands0[1, k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)])])
    sigma_bands = sigma_bands0[:, :, :n_sigma_bands + 1]
    pi_bands = pi_bands0[:, :, :n_pi_bands + 1]
    
    return sigma_bands, pi_bands

def bandstructure_plot(sigma_bands, pi_bands, n_kpoints: int = 0, whichspin: int = 0, energy_range = [-7, 7], gamma: float = .01, de: float = .05, aspect_ratio: float = .1, mode: str = "symmetries", second_BZ: float = .2):
    n_sigma_bands = len(sigma_bands[0, 0])
    n_pi_bands = len(pi_bands[0, 0])

    fig, ax = plt.subplots()
    fig.set_size_inches([4.8, 3.6], forward = True)
    plt.tight_layout()
    ax.set_xticks([-.75, -.5, 0, .5])
    ax.set_xticklabels(["pDOS", "-X", "Γ", "X"])
    ax.set_aspect(aspect_ratio)
    ax.axvline(x = -.5, color = 'k')
    ax.axvline(x = 0, color = 'gray')
    ax.axvline(x = .5, color = 'gray')

    # For simple uninterrupted bands, use pyplot
    k_points = np.linspace(0, .5, n_kpoints)
    if mode == "spins":
        [ax.plot(-k_points, sigma_bands[0, :, i, 0], c = dark_green_semitransparent) for i in range(n_sigma_bands)] # Plot the sigma bands to the left of Gamma (Spin 0, all k points, band by band, eigenenergy slot)
        [ax.plot(k_points, sigma_bands[1, :, i, 0], c = dark_green_semitransparent) for i in range(n_sigma_bands)] # Plot them transparently to the right of Gamma
        [ax.plot(1 - k_points, sigma_bands[1, :, i, 0], c = dark_green_semitransparent) for i in range(n_sigma_bands)] # Plot a tiny section in the second BZ
    else:
        [ax.plot(-k_points, sigma_bands[0, :, i, 0], c = dark_green) for i in range(n_sigma_bands)] # Plot the sigma bands to the left of Gamma (Spin 0, all k points, band by band, eigenenergy slot)
    
    # For projections, use line collection
    for band in range(n_pi_bands):
        if mode == "spins": segments = [[[k_points[i], pi_bands[1, i, band, 0]], [k_points[i + 1], pi_bands[1, i + 1, band, 0]]] for i in range(n_kpoints - 1)]
        else: segments = [[[k_points[i], pi_bands[0, i, band, 0]], [k_points[i + 1], pi_bands[0, i + 1, band, 0]]] for i in range(n_kpoints - 1)]
        colors = []
        for i in range(n_kpoints - 1):
            avg_coeffs = np.average(pi_bands[whichspin, i:i + 2, band], axis = 0)
            colors.append((.8 * avg_coeffs[1], 0, .7 * (1 - avg_coeffs[1]), 1))
        lc = LineCollection(segments = segments, colors = colors)
        ax.add_collection(lc)
        if mode == "spins":
            segments = [[[-k_points[i], pi_bands[0, i, band, 0]], [-k_points[i + 1], pi_bands[0, i + 1, band, 0]]] for i in range(n_kpoints - 1)]
            lc = LineCollection(segments = segments, colors = colors)
            ax.add_collection(lc)
        
        # Add a tiny slither in the second BZ
        segments = [[[1 - k_points[i], pi_bands[1, i, band, 0]], [1 - k_points[i + 1], pi_bands[1, i + 1, band, 0]]] for i in range(n_kpoints - 1)]
        lc = LineCollection(segments = segments, colors = colors)
        ax.add_collection(lc)
    
    # Calculate the projected DOS
    e_list = np.arange(energy_range[0], energy_range[1], de)
    dos_sigma_up = np.zeros_like(e_list, dtype = float)
    dos_sigma_down = np.zeros_like(e_list, dtype = float)
    dos_pi_up = np.zeros_like(e_list, dtype = float)
    dos_pi_down = np.zeros_like(e_list, dtype = float)
    gamma2 = gamma ** 2

    for i in range(len(e_list)): # Loop over energy points
        en_diffs2 = (np.array([[(sigma_bands[0, k_point, band, 0] - e_list[i]) ** 2 for band in range(n_sigma_bands)] for k_point in range(n_kpoints)])).flatten() # Calculate the distances between the energy point and the eigenenergies
        en_diffs2_sel = [en_diff2 for en_diff2 in en_diffs2 if en_diff2 < 90] # Ditch eigenenergies that are too far away and have a negligible contribution to the DOS
        dos_sigma_up[i] = sum([(gamma / np.pi) * (1  / (en_diff2 ** 2 + gamma2)) for en_diff2 in en_diffs2_sel]) # Sum over Lorentzian functions to determine the DOS as the energy point
        en_diffs2 = (np.array([[(pi_bands[0, k_point, band, 0] - e_list[i]) ** 2 for band in range(n_pi_bands)] for k_point in range(n_kpoints)])).flatten() # Do this again, but now for the pi states
        en_diffs2_sel = [en_diff2 for en_diff2 in en_diffs2 if en_diff2 < 90]
        dos_pi_up[i] = sum([(gamma / np.pi) * (1  / (en_diff2 ** 2 + gamma2)) for en_diff2 in en_diffs2_sel])
    
    for i in range(len(e_list)): # Loop over energy points
        en_diffs2 = (np.array([[(sigma_bands[1, k_point, band, 0] - e_list[i]) ** 2 for band in range(n_sigma_bands)] for k_point in range(n_kpoints)])).flatten() # Calculate the distances between the energy point and the eigenenergies
        en_diffs2_sel = [en_diff2 for en_diff2 in en_diffs2 if en_diff2 < 90] # Ditch eigenenergies that are too far away and have a negligible contribution to the DOS
        dos_sigma_down[i] = sum([(gamma / np.pi) * (1  / (en_diff2 ** 2 + gamma2)) for en_diff2 in en_diffs2_sel]) # Sum over Lorentzian functions to determine the DOS as the energy point
        en_diffs2 = (np.array([[(pi_bands[1, k_point, band, 0] - e_list[i]) ** 2 for band in range(n_pi_bands)] for k_point in range(n_kpoints)])).flatten() # Do this again, but now for the pi states
        en_diffs2_sel = [en_diff2 for en_diff2 in en_diffs2 if en_diff2 < 90]
        dos_pi_down[i] = sum([(gamma / np.pi) * (1  / (en_diff2 ** 2 + gamma2)) for en_diff2 in en_diffs2_sel])
    
    norm_dos = 2 * np.max(dos_sigma_up + dos_pi_up)
    dos_pi_up /= norm_dos
    dos_sigma_up /= norm_dos
    dos_pi_down /= norm_dos
    dos_sigma_down /= norm_dos
    
    if mode == "spins":
        ax.fill_betweenx(e_list, - 0.75 - .5 * (dos_pi_up + dos_sigma_up), - 0.75 - .5 * dos_pi_up, color = dark_green_semitransparent) # Color the sigma DOS blue
        ax.fill_betweenx(e_list, - 0.75 - .5 * dos_pi_up, - 0.75, color = dark_purple) # Color the pi DOS red
        ax.fill_betweenx(e_list, - 0.75 + .5 * (dos_pi_down + dos_sigma_down), - 0.75 + .5 * dos_pi_down, color = dark_green_semitransparent) # Color the sigma DOS blue
        ax.fill_betweenx(e_list, - 0.75 + .5 * dos_pi_down, - 0.75, color = dark_cyan) # Color the pi DOS red
    else:
        ax.fill_betweenx(e_list, - 0.5 - dos_pi_up - dos_sigma_up, - 0.5 - dos_pi_up, color = dark_green_semitransparent) # Color the sigma DOS blue
        ax.fill_betweenx(e_list, - 0.5 - dos_pi_up, - 0.5, color = dark_purple) # Color the pi DOS red
    ax.set_xbound([-1, .5 + second_BZ / 2])
    ax.set_ybound(energy_range)
    #plt.savefig(folder_name + "BS_full.svg")
    return fig

def get_Fourier_cube(wavecar_data, k_point: int = 0, band: int = 0, whichspin: int = 0, verbose: bool = True):
    if k_point > wavecar_data.nk - 1:
        print(f"Error! Selected k-point ({k_point}) out of range (total number of k-points: {wavecar_data.nk})")
        return [[[]]]
    if band > wavecar_data.nb - 1:
        print(f"Error! Selected band index ({band}) out of range (total number of bands: {wavecar_data.nb})")
        return [[[]]]

    spin_polarized = bool(wavecar_data.spin - 1)
    upordown = "up"
    if whichspin == 1: upordown = "down"

    # Read wavecar data at the k-point
    g_points = wavecar_data.Gpoints[k_point]
    if spin_polarized: eigs = wavecar_data.band_energy[whichspin][k_point]
    else: eigs = wavecar_data.band_energy[k_point]
    
    nx_max = int(np.max(np.array([g_points[i][0] for i in range(len(g_points))])))
    ny_max = int(np.max(np.array([g_points[i][1] for i in range(len(g_points))])))
    nz_max = int(np.max(np.array([g_points[i][2] for i in range(len(g_points))])))

    # Read data for the specific band
    if spin_polarized: wave_band = wavecar_data.coeffs[whichspin][k_point][band]
    else: wave_band = wavecar_data.coeffs[k_point][band]

    # Check if the wfn has more weight on nx = 0 (meaning it is symmetric wrt reflection in the yz plane) or on nx = 1 (antisymmetric wrt reflection in the yz plane)
    g_points_list = [[0., 1., 1.], [1., 1., 1.]] # All wfns have weight on ny = 1 and nz = 1 due to the modulation incurred from the vacuum regions
    indices = [(int(np.where(np.all(g_points == g_point, axis = 1))[0][0])) for g_point in g_points_list]
    coeffs = np.abs(np.array([wave_band[index] for index in indices]))
    if coeffs[0] > coeffs[1]: nx = 0
    else: nx = 1

    # Check the wfn sigma/pi symmetry
    g_points_list = [[nx * 1., 1., 1.], [nx * 1., 1., -1.]] # Opposite k-points (reflected in z) are retrieved at nx = 0 and nx = 1
    indices = [(int(np.where(np.all(g_points == g_point, axis = 1))[0][0])) for g_point in g_points_list]
    coeffs = np.array([wave_band[index] for index in indices])
    if np.abs(np.diff(coeffs)) < 10 ** (-6): symz = 0
    else: symz = 1
    nz = symz # If the wfn is a sigma orbital, focus only on wfns with nz = 0; otherwise, focus on wfns with nz = 1

    if verbose:
        if nz == 0: print(f"Orbital number {band} with spin {whichspin} (spin {upordown}) at k-point {k_point} and energy {eigs[band][0]} eV. This orbital has sigma symmetry")
        else: print(f"Orbital number {band} with spin {whichspin} (spin {upordown}) at k-point {k_point} and energy {eigs[band][0]} eV. This orbital has pi symmetry")
    
    # Read in the coefficients
    Fourier_cube = np.zeros((2 * nx_max + 1, 2 * ny_max + 1, 2 * nz_max + 1), dtype = complex)
    for i in range(len(g_points)):
        nx, ny, nz = [int(g_points[i, dim]) for dim in range(3)]
        coeff = wave_band[i]
        Fourier_cube[nx, ny, nz] = coeff

    return Fourier_cube

def get_wfn(wavecar_data, k_point: int = 0, band: int = 0, whichspin: int = 0, normalize: bool = True, phase_autozero: bool = True, n_bins: int = 100, verbose: bool = True):
    Fourier_cube = get_Fourier_cube(wavecar_data, k_point = k_point, band = band, whichspin = whichspin, verbose = verbose)
    if len(Fourier_cube) < 2: return [[[]]]
    wfn_cube = np.fft.ifftn(Fourier_cube)
    wfn2_cube = np.abs(wfn_cube) ** 2
    max_value = np.max(wfn2_cube)
    norm_fac = np.sqrt(max_value)
    
    if normalize and max_value > .0000000000001:
        wfn_cube /= (norm_fac + .0000000000001)
        wfn2_cube = np.abs(wfn_cube) ** 2
    
    wfnarg_cube = np.angle(wfn_cube)
    if phase_autozero:
        arg_histogram = np.histogram(wfnarg_cube.flatten(), bins = n_bins, range = [-np.pi, np.pi])
        max_index = np.where(arg_histogram[0] == np.max(arg_histogram[0]))[0][0]
        max_represented_phase = max_index * (2 * np.pi) / n_bins - np.pi
        wfnarg_cube = wfnarg_cube - max_represented_phase

    return wfn_cube, wfn2_cube, wfnarg_cube

def wfn_slice(wavecar_data, wfn_cube, slice_height: float = 5):
    lat_vec_z = wavecar_data.a[2, 2]
    grid_z = np.shape(wfn_cube)[2]
    Angstrom_per_voxel_z = lat_vec_z / grid_z
    slice_index = int(round(slice_height / Angstrom_per_voxel_z))
    wfn_rect = np.flip(wfn_cube[:, :, slice_index])

    return slice_index, wfn_rect

def orbital_render(wfn, phase_offset: float = -.4, contrast: float = 1., brightness: float = 1., upsample: float = 3.):
    wfn_arg = np.angle(wfn)
    wfn2 = np.abs(wfn) ** 2
    orbital_img = np.zeros((np.shape(wfn)[0], np.shape(wfn)[1], 4), dtype = float)

    for j in range(len(orbital_img[0])):
        for i in range(len(orbital_img)):
            phase = wfn_arg[i, j] + phase_offset
            amplitude = wfn2[i, j]
            RGBfactors = .499 * np.array([np.cos(phase), np.cos(phase + 2 * np.pi / 3), np.cos(phase + 4 * np.pi / 3)]) * contrast + .5 * brightness
            orbital_img[i, j] = [RGBfactors[0], RGBfactors[1], RGBfactors[2], amplitude]
    
    return np.flip(orbital_img, axis = 0) #np.clip(zoom(orbital_img, (upsample, upsample)), a_min = 0, a_max = 1)

def orbital_plots(wavecar_data, k_point: int = 0, band: int = 0, height: float = 5., cells = [3, 1], zero_phase: float = 2.6, whichspin: int = 0, brightness: float = 1, normalize_slice: bool = True, plot_type: str = "density", threshold: float = .1, zoom_factor: int = 3, verbose: bool = True, bonds_collection = False, atoms_collection = False):
    folder_name = os.path.dirname(wavecar_data.filename) + "/" # To do: use the HDF5 file instead of the wavecar
    file_name_struc = folder_name + "CONTCAR"
    structure = Structure.from_file(file_name_struc)
    height_relative_pm = round(100 * (height - np.average(structure.frac_coords[:, 2]) * structure.lattice.c))

    wfn_folder_name = folder_name + "wfn_data"
    try:
        os.makedirs(wfn_folder_name, exist_ok=True)
        if verbose: print(f"Directory {wfn_folder_name} created successfully or already exists.")
    except OSError as e: print(f"Error creating directory {wfn_folder_name}: {e}")
    plot_basename = wfn_folder_name + "/orb_" + str(band) + "_k_" + str(k_point) + "_spin_" + str(whichspin) + "_height_" + str(height_relative_pm) + "_pm_"

    Fourier_cube = get_Fourier_cube(wavecar_data, k_point = k_point, band = band, whichspin = whichspin, verbose = True)
    wfn_cube = np.fft.ifftn(Fourier_cube)
    wfn2_cube = np.abs(wfn_cube) ** 2
    max_value = np.max(wfn2_cube)
    norm_fac = np.sqrt(max_value)
    if max_value > .0000000000001:
        wfn_cube /= (norm_fac + .0000000000001)
        wfn2_cube = np.abs(wfn_cube) ** 2
    wfnarg_cube = np.angle(wfn_cube)

    arg_histogram = np.histogram(wfnarg_cube.flatten(), bins = 100, range = [-np.pi, np.pi])
    max_index = np.where(arg_histogram[0] == np.max(arg_histogram[0]))[0][0]
    max_represented_phase = max_index * (2 * np.pi) / 100 - np.pi
    wfnarg_cube = wfnarg_cube - max_represented_phase

    slice_index, wfn_rect = wfn_slice(wavecar_data, wfn_cube, slice_height = height) # Get 2D slices of the 3D data
    slice_index, wfn2_rect = wfn_slice(wavecar_data, wfn2_cube, slice_height = height)
    slice_index, wfnarg_rect = wfn_slice(wavecar_data, wfnarg_cube, slice_height = height)

    # Calculate the standard orbital
    if normalize_slice: wfn2_rect /= np.max(wfn2_rect)

    arg_img = np.array([[colorfunction_hue(wfnarg_rect[i, j] + zero_phase) for i in range(len(wfnarg_rect))] for j in range(len(wfnarg_rect[0]))]) # Create the color image from the wavefunction arguments
    orbital_img = np.array([[(arg_img[j, i, 0], arg_img[j, i, 1], arg_img[j, i, 2], wfn2_rect[i, j]) for i in range(len(wfn2_rect))] for j in range(len(wfn2_rect[0]))])

    zoomed_orbital = np.clip(zoom(orbital_img, (zoom_factor, zoom_factor, 1)), a_min = 0, a_max = 1)
    zoomed_orbital = np.vstack([np.hstack([zoomed_orbital for _ in range(cells[0])]) for _ in range(cells[1])])
    
    fig, ax = plt.subplots()
    ax.imshow(zoomed_orbital, extent = [0, structure.lattice.a * cells[0], 0, structure.lattice.b * cells[1]], zorder = 0)

    if type(bonds_collection) == PatchCollection:
        ax.add_collection(bonds_collection)
    if type(atoms_collection) == PatchCollection:
        ax.add_collection(atoms_collection)
    ax.set_xbound([0, cells[0] * structure.lattice.a])
    ax.set_ybound([0, cells[1] * structure.lattice.b])
    plt.tight_layout()
    if type(bonds_collection) == PatchCollection: fig.savefig(plot_basename + "wfn.svg")
    plt.show()
    plt.imsave(plot_basename + "wfn.png", zoomed_orbital)

    # Calculate the orbital as gauged with a pi-wave tip (CO tip)
    dwfn_rect_dx = convolve2d(wfn_rect, np.array([[1, -1], [1, -1]], dtype = float), boundary = "wrap")
    diwfn_rect_dy = convolve2d(wfn_rect, np.array([[1, 1], [-1, -1]], dtype = float), boundary = "wrap")
    dwfn_rect_dxdy = dwfn_rect_dx + 1j * diwfn_rect_dy
    piwfn2_rect = np.abs(dwfn_rect_dxdy) ** 2
    piwfn2_rect /= np.max(piwfn2_rect)
    piwfnarg_rect = np.angle(dwfn_rect_dxdy)
    piarg_img = np.array([[colorfunction_hue(piwfnarg_rect[i, j] + zero_phase) for i in range(len(piwfnarg_rect))] for j in range(len(piwfnarg_rect[0]))]) # Create the color image from the wavefunction arguments
    piorbital_img = np.array([[(piarg_img[j, i, 0], piarg_img[j, i, 1], piarg_img[j, i, 2], piwfn2_rect[i, j]) for i in range(len(piwfn2_rect))] for j in range(len(piwfn2_rect[0]))])

    zoomed_piorbital = np.clip(zoom(piorbital_img, (zoom_factor, zoom_factor, 1)), a_min = 0, a_max = 1)
    zoomed_piorbital = np.vstack([np.hstack([zoomed_piorbital for _ in range(cells[0])]) for _ in range(cells[1])])

    plt.imshow(zoomed_piorbital, extent = [0, structure.lattice.a * cells[0], 0, structure.lattice.b * cells[1]], zorder = 0)
    plt.show()
    plt.imsave(plot_basename + "wfn_COtip.png", zoomed_piorbital)

def eigenstates_to_hdf(wavecar_data, slice_height: float = 7., midgap_energy: float = 0, energy_range = [-2., 2.]):
    folder_name = os.path.dirname(wavecar_data.filename)
    n_kpoints = wavecar_data.nk
    n_bands = wavecar_data.nb
    lat_vec_z = wavecar_data.a[2, 2]
    spin_polarized = bool(wavecar_data.spin - 1)
    en_min = np.min(energy_range)
    en_max = np.max(energy_range)

    if spin_polarized: all_eigs = np.array(wavecar_data.band_energy)
    else: all_eigs = np.array([wavecar_data.band_energy, wavecar_data.band_energy])
    all_eigs[:, :, :, 0] -= midgap_energy

    band_select = []
    for band in range(n_bands):
        min_up, max_up = [np.min(all_eigs[0, :, band, 0]), np.max(all_eigs[0, :, band, 0])]
        min_down, max_down = [np.min(all_eigs[1, :, band, 0]), np.max(all_eigs[1, :, band, 0])]
        band_select.append(max_up > en_min and min_up < en_max or max_down > en_min and min_down < en_max)

    band_range = [np.min(np.where(band_select)), np.max(np.where(band_select))]

    wfn_cube = get_wfn(wavecar_data, k_point = 0, band = 0, normalize = False, phase_autozero = True, n_bins = 100, verbose = False)[0]
    grid_z = np.shape(wfn_cube)[2]
    Angstrom_per_voxel_z = lat_vec_z / grid_z
    slice_index = int(round(slice_height / Angstrom_per_voxel_z))
    wfn_shape = np.shape(wfn_cube)
    eigenstates_array = np.zeros((2, n_kpoints, band_range[1] + 1 - band_range[0], wfn_shape[0], wfn_shape[1]), dtype = complex)

    for whichspin in range(2):
        for band in range(band_range[0], band_range[1] + 1):
            for k_point in range(n_kpoints):
                wfn_cube = get_wfn(wavecar_data, k_point = k_point, band = band, whichspin = whichspin, normalize = False, phase_autozero = True, n_bins = 100, verbose = False)[0]
                norm_fac = np.sum(np.abs(wfn_cube) ** 2)
                wfn_cube /= np.sqrt(norm_fac)
                wfn_rect = np.flip(wfn_cube[:, :, slice_index])
                eigenstates_array[whichspin, k_point, band - band_range[0]] = wfn_rect

    with h5.File(folder_name + "/eigenstates_at_h=" + str(int(np.floor(slice_height * 100))) + "_pm.hdf5", "w") as f:
        f.create_dataset("eigenstates", data = eigenstates_array)
        f.create_dataset("eigenstate energies", data = all_eigs[:, :, band_range[0]:band_range[1] + 1])

def dIdV_maps(folder_name: str, energies, zoom_factor: float = 3, gamma: float = .05, range_rescale: float = 1, pi_range_rescale: float = 1):
    file_list = []
    for entry in os.listdir(folder_name): # Find the HDF5 file
        if os.path.isfile(os.path.join(folder_name, entry)) and entry.endswith(".hdf5"):
            file_list.append(os.path.join(folder_name, entry))
    if len(file_list) < 1:
        print("No hdf5 file found!")
        return
    hdf5_file = file_list[0] # Create an output folder for the maps
    output_folder_name = os.path.dirname(folder_name) + "/dIdV_maps"
    try: os.makedirs(output_folder_name, exist_ok = True)
    except:
        print("Error creating the output folder!")
        return
    
    with h5.File(hdf5_file, "r") as f: # Read the eigenstates from the HDF5 file
        dataset = f["eigenstates"]
        wfns = dataset[:]
        dataset = f["eigenstate energies"]
        all_eigs = dataset[:]

    all_eigs = all_eigs[:, :, :, 0].flatten() # Put all eigenenergies in a row
    wfn2s = np.zeros_like(wfns, dtype = float)
    piwfn2s = np.zeros_like(wfns, dtype = float)
    wfns_shape = np.shape(wfns)
    n_spins = wfns_shape[0]
    n_kpoints = wfns_shape[1]
    n_bands = wfns_shape[2]

    for whichspin in range(n_spins): # Read the wavefunction of each eigenstate and compute its orbital (absolute square)
        for k_point in range(n_kpoints):
            for band in range(n_bands):
                wfn = wfns[whichspin, k_point, band] # Regular complex real-space wavefunction
                wfn2 = np.abs(wfn) ** 2 # Orbital (absolute square)
                wfn2s[whichspin, k_point, band] = wfn2

                sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype = int)
                sobel_y = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype = int)
                dwfn_dx = convolve2d(wfn, sobel_x, boundary = "wrap", mode = "same") # Compute the x- and y derivatives of the wfn by convolution with Sobel kernels
                dwfn_dy = convolve2d(wfn, sobel_y, boundary = "wrap", mode = "same")
                dwfn_dxdy = dwfn_dx + 1j * dwfn_dy
                piwfn2 = np.abs(dwfn_dxdy) ** 2
                piwfn2s[whichspin, k_point, band] = piwfn2

    wfn2s = wfn2s.reshape(np.prod(wfns_shape[:3]), wfns_shape[3], wfns_shape[4]) # Put all orbitals in a row
    piwfn2s = piwfn2s.reshape(np.prod(wfns_shape[:3]), wfns_shape[3], wfns_shape[4]) # Put all orbitals in a row

    gamma2 = gamma ** 2 # Loop over all the target energies and compute the dI/dV map by summing all orbitals weighted by their Lorentzian weight
    maps = []
    maps_pi = []
    for en in energies:
        en_diffs2 = (all_eigs - en) ** 2
        weights = [(gamma / np.pi) * (1  / (en_diff2 ** 2 + gamma2)) for en_diff2 in en_diffs2]
        weighted_map = sum([weights[i] * wfn2s[i] for i in range(len(weights))])
        maps.append(weighted_map)

        weighted_map_pi = sum([weights[i] * piwfn2s[i] for i in range(len(weights))])
        maps_pi.append(weighted_map_pi)
    maps = np.array(maps)
    maps_pi = np.array(maps_pi)

    max_values = [np.max(maps[i]) for i in range(len(maps))] # Normalize the maps with respect to the maximum value found in all maps
    norm_fac = np.max(max_values)
    maps /= norm_fac
    maps *= range_rescale # Change the contrast by another factor if desired (maybe in the case of outliers that reduce the dynamic range of the contrast)

    max_values = [np.max(maps_pi[i]) for i in range(len(maps_pi))] # Normalize the maps with respect to the maximum value found in all maps
    norm_fac = np.max(max_values)
    maps_pi /= norm_fac
    maps_pi *= pi_range_rescale # Change the contrast by another factor if desired (maybe in the case of outliers that reduce the dynamic range of the contrast)

    for i in range(len(energies)): # Save the maps to the output folder
        plot_basename = output_folder_name + "/dIdVmap_@_" + str(int(round(energies[i] * 1000))) + "_meV"
        img = np.clip(zoom(maps[i].T, (zoom_factor, zoom_factor)), a_min = 0, a_max = 1)
        plt.imsave(plot_basename + ".png", img, vmin = 0, vmax = 1, cmap = "gray")

        plot_basename = output_folder_name + "/CO_tip_dIdVmap_@_" + str(int(round(energies[i] * 1000))) + "_meV"
        img = np.clip(zoom(maps_pi[i].T, (zoom_factor, zoom_factor)), a_min = 0, a_max = 1)
        plt.imsave(plot_basename + ".png", img, vmin = 0, vmax = 1, cmap = "gray")

def grid_plot(matrix, return_type = "show"):
    fig, ax = plt.subplots()
    ax.pcolormesh(np.flip(matrix, axis = 0), cmap = "seismic", vmin = -2, vmax = 2)
    ax.set_aspect(1)
    ax.set_xticks(range(len(matrix[0])))
    ax.tick_params(axis = "x", labelbottom = False)
    ax.set_yticks(range(len(matrix)))
    ax.tick_params(axis = "y", labelleft = False)
    ax.grid(True)
    fig.set_size_inches(16, 4)
    if return_type == "show":
        plt.show()
    else: return fig

np.set_printoptions(suppress = True)

Simple visualization tool

In [ ]:
file_name = "C:/DFT/Primitive_GNRs/4-AGNR/CONTCAR"

structure = Structure.from_file(file_name)
atoms = structure.sites
n_atoms = len(atoms)
n_elec = calculate_electrons(structure)

print("Number of atoms:", n_atoms)
print("Number of electrons:", n_elec)
structure_plot(structure, dimensions = 2, contrast = .5, brightness = .75)

In [ ]:
structure = Structure.from_file("C:\\DFT\\Odd_rings\\6-notched_4-ZGNR\\4-notched_6-ZGNR_POSCAR")
structure_plot(structure)
xyz_list = structure.cart_coords
sites_list = structure.sites

In [ ]:
atoms_left = np.where([xyz_list[i, 0] < 1 for i in range(len(xyz_list))])[0]
structure_cropped = structure.copy()
structure_cropped.remove_sites(atoms_left)
lat_vecs = structure_cropped.lattice.matrix

delta_x = 1
new_lattice = Lattice.from_parameters(structure_cropped.lattice.a - delta_x, structure_cropped.lattice.b, structure_cropped.lattice.c, 90, 90, 90)
structure_cropped.translate_sites(range(len(structure_cropped.sites)), .5 * lat_vecs[0] + [-np.mean(structure_cropped.cart_coords[:, 0]) - .5 * delta_x, 0, 0], frac_coords = False)
new_structure = Structure(new_lattice, structure_cropped.species, structure_cropped.cart_coords, coords_are_cartesian = True)
structure_plot(new_structure)
new_structure.to_file("C:\\DFT\\Odd_rings\\6-notched_4-ZGNR\\4-notched_POSCAR", fmt = "poscar")

Band structure from PROCAR file

In [ ]:
N_ribbon = 2

#folder_name = "C:/DFT/" + str(N_ribbon) + "-AGNR/" + str(N_ribbon) + "-AGNR_primitive/"
folder_name = "C:/DFT/" + str(N_ribbon) + "-AGNR/"
folder_name = "C:/DFT/4-AGNR/4-AGNR_8_N_row2/"

file_name = folder_name + "CONTCAR" # Get the CONTCAR for visualization and determination of the atoms (species and coordinates)

n_bands, n_kpoints, n_ions, n_noH, n_orbitals, spin_polarized, orbital_list, pz_index, luco_index, hoco_energy, luco_energy, midgap_energy, band_gap, procar_data = procar_loader(folder_name)

In [20]:
midgap_to_zero = True

sigma_bands_up0 = np.zeros((n_kpoints, n_bands, 5), dtype = float) # Initialize arrays for the sigma and pi bands
pi_bands_up0 = sigma_bands_down0 = pi_bands_down0 = np.zeros_like(sigma_bands_up0) # The arrays are initialized too large but will later be truncated to fit the number of bands

for k_point in range(n_kpoints): # Loop over all k-points
    if spin_polarized:
        up_occupancies = procar_data.occupancies[Spin.up][k_point] # Read the orbital occupations
        down_occupancies = procar_data.occupancies[Spin.down][k_point] # Read spin down occupancies only if the calculation is spin-polarized
    else:
        up_occupancies = procar_data.occupancies[Spin.up][k_point] / 2
        down_occupancies = up_occupancies

    if midgap_to_zero: up_data = procar_data.eigenvalues[Spin.up][k_point] - midgap_energy # Read the eigenvalues
    else: up_data = procar_data.eigenvalues[Spin.up][k_point]
    up_proj_matrix = procar_data.data[Spin.up][k_point] # Read the projections
    down_data = up_data
    down_proj_matrix = up_proj_matrix
    if spin_polarized:
        if midgap_to_zero: down_data = procar_data.eigenvalues[Spin.down][k_point] - midgap_energy
        else: down_data = procar_data.eigenvalues[Spin.down][k_point]
        down_proj_matrix = procar_data.data[Spin.down][k_point]

    n_sigma_up = 0
    n_pi_up = 0
    n_sigma_down = 0
    n_pi_down = 0

    for band in range(n_bands): # Loop over the bands
        eigenenergy = up_data[band]
        occupancy = up_occupancies[band]
        norm_fac = np.sum(up_proj_matrix[band]) + .000000001 # Normalize the total of the projection matrix. The offset is to prevent accidental divisions by zero
        noH_ions = sum([up_proj_matrix[band][atom] for atom in range(n_noH)]) # Sum projections over atoms that are not hydrogen
        H_ions = sum(sum([up_proj_matrix[band][atom] for atom in range(n_noH, n_ions)])) / norm_fac # Sum projections over atoms that are hydrogen
        noH_ions_pz = noH_ions[pz_index] / norm_fac # The projection found at the pz_index'th slot is the projection on the pz orbitals
        noH_ions_other = sum([noH_ions[i] for i in list(set(range(n_orbitals)).symmetric_difference(set([2])))]) / norm_fac # The rest of the orbitals are not pz

        if noH_ions_pz < noH_ions_other + H_ions:
            sigma_bands_up0[k_point, n_sigma_up, 0] = eigenenergy
            sigma_bands_up0[k_point, n_sigma_up, 1] = occupancy
            n_sigma_up += 1
        else:
            pi_bands_up0[k_point, n_pi_up, 0] = eigenenergy
            pi_bands_up0[k_point, n_pi_up, 1] = occupancy
            n_pi_up += 1

        energy = down_data[band]
        occupancy = down_occupancies[band]
        norm_fac = np.sum(down_proj_matrix[band]) + .000000001
        noH_ions = sum([down_proj_matrix[band][atom] for atom in range(n_noH)])
        H_ions = sum(sum([down_proj_matrix[band][atom] for atom in range(n_noH, n_ions)])) / norm_fac
        noH_ions_pz = noH_ions[pz_index] / norm_fac
        noH_ions_other = sum([noH_ions[i] for i in list(set(range(n_orbitals)).symmetric_difference(set([2])))]) / norm_fac

        if noH_ions_pz < noH_ions_other + H_ions:
            sigma_bands_down0[k_point, n_sigma_down, 0] = eigenenergy
            sigma_bands_down0[k_point, n_sigma_down, 1] = occupancy
            n_sigma_down += 1
        else:
            pi_bands_down0[k_point, n_pi_down, 0] = eigenenergy
            pi_bands_down0[k_point, n_pi_down, 1] = occupancy
            n_pi_down += 1

n_sigma_bands_up = np.min(np.array([np.where(np.diff(sigma_bands_up0[k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)])) # Determine where to crop the band arrays
n_pi_bands_up = np.min(np.array([np.where(np.diff(pi_bands_up0[k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)]))
sigma_bands_up = np.array([[sigma_bands_up0[:, i, 0], sigma_bands_up0[:, i, 1], np.ones_like(sigma_bands_up0[:, i, 0])] for i in range(n_sigma_bands_up)])
pi_bands_up = np.array([[pi_bands_up0[:, i, 0], pi_bands_up0[:, i, 1], np.ones_like(sigma_bands_up0[:, i, 0])] for i in range(n_pi_bands_up)])

n_sigma_bands_down = np.min(np.array([np.where(np.diff(sigma_bands_down0[k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)])) # Determine where to crop the band arrays
n_pi_bands_down = np.min(np.array([np.where(np.diff(pi_bands_down0[k_point, :, 0]) < 0.)[0][0] for k_point in range(n_kpoints)]))
sigma_bands_down = np.array([[sigma_bands_down0[:, i, 0], sigma_bands_down0[:, i, 1], np.ones_like(sigma_bands_down0[:, i, 0])] for i in range(n_sigma_bands_down)])
pi_bands_down = np.array([[pi_bands_down0[:, i, 0], pi_bands_down0[:, i, 1], np.ones_like(sigma_bands_down0[:, i, 0])] for i in range(n_pi_bands_down)])

Load WAVECAR and CONTCAR files

In [ ]:
N_GNR = 12
GNR_orientation = "A" # "A" for armchair or "Z" for zigzag
defect_type = "Be" # Swap to something nonsensical for primitive cell calculations
defect_row = 3

if defect_type != "N" and defect_type != "B": folder_name = "C:/DFT/Primitive_GNRs/" + str(N_GNR) + "-" + GNR_orientation + "GNR/" # Primitive GNRS
else:
    folder_name = "C:/DFT/" + str(N_GNR) + "-" + GNR_orientation + "GNR_defects/"
    defect_folders = [item for item in os.listdir(folder_name) if os.path.isdir(folder_name + item)]
    digits = list(filter(str.isdigit, defect_folders[0]))
    if len(digits) == 4: supercell_length = int(digits[1] + digits[2])
    else: supercell_length = int(digits[1])
    folder_name = "C:/DFT/" + str(N_GNR) + "-" + GNR_orientation + "GNR_defects/" + str(N_GNR) + "-" + GNR_orientation + "GNR_" + str(supercell_length) + "_" + str(defect_type) + "_row" + str(defect_row) + "/"

#folder_name = "C:\\DFT\\Odd_rings\\4-notched_4-ZGNR\\"

print("Extracting data from: " + folder_name)
n_bands, n_kpoints, spin_polarized, luco_index, hoco_energy, luco_energy, midgap_energy, band_gap, wavecar_data, structure = wavecar_loader(folder_name)
structure_plot(structure, dimensions = 2)
n_atoms = len(structure.sites)
n_elec = calculate_electrons(structure)

print("Number of atoms:", n_atoms)
print("Number of electrons:", n_elec)
print(luco_index)

Save the structure in svg format

In [ ]:
cells = [1, 1]

plot1 = structure_plot(structure, supercell = (cells[0], cells[1], 1), dimensions = 2, atom_size = .2, bond_size = 1.2, contrast = 1, brightness = .5, return_type = "figure")
plot1.savefig(folder_name + "struc_2D.svg")
plt.show()
plot2 = structure_plot(structure, supercell = (cells[0], cells[1], 1), dimensions = 2, atom_size = .1, bond_size = .5, contrast = .5, brightness = .75, return_type = "figure")
plot2.savefig(folder_name + "struc_2D_light.svg")
plt.show()

In [ ]:
midgap_energy_pristine = wavecar_loader("C:/DFT/Primitive_GNRs/" + str(N_GNR) + "-" + GNR_orientation + "GNR/")[6]

In [13]:
midgap_energy_pristine = midgap_energy

Calculate the band structure from the WAVECAR file

In [14]:
sigma_bands, pi_bands = get_bands(wavecar_data, midgap_energy_pristine)

Plot the band structure from the WAVECAR file

In [ ]:
#energy_range = [-7, 7]
energy_range = [-2.5, 2.5]
de = .02 # Resolution for DOS calculation
gamma = .005 # Lorentzian width for broadening
whichspin = 1

bs = bandstructure_plot(sigma_bands, pi_bands, n_kpoints, whichspin = whichspin, energy_range = energy_range, gamma = gamma, de = de, aspect_ratio = .2 * 3.5 / 2.5, mode = "symmetries", second_BZ = 0)
bs.savefig(folder_name + "BS_full_2500meVrange.svg")
plt.show()

Save the eigenstates (real-space wavefunctions calculated at a certain height) within a certain energy range for all spins, k-points and bands

In [ ]:
eigenstates_to_hdf(wavecar_data, slice_height = 7., midgap_energy = midgap_energy_pristine, energy_range = [-3, 3])

Simulate STM dI/dV maps

In [25]:
dIdV_maps(folder_name, energies = np.arange(-1.4, 1.4 + .2, .2), zoom_factor = 3, gamma = .05, range_rescale = 2.4, pi_range_rescale = 2)

Orbital plotting

In [ ]:
k_point = 0
band = 21
whichspin = 0

height = 7.
zero_phase = -.4
zoom_factor = 3
cells = [1, 1]




#for band in range(luco_index - 10, luco_index + 10):
#    orbital_plots(wavecar_data, k_point = k_point, band = band, height = height, cells = cells, zero_phase = zero_phase, zoom_factor = zoom_factor)

Miscellaneous (bouncing electron packets)

In [ ]:
x_values = np.linspace(-4, 4, 240)
y_values = np.linspace(-4, 4, 240)
X, Y = np.meshgrid(x_values, y_values)

r1_0 = np.array([0, 0], dtype = float)
k1 = np.array([1, 0], dtype = float)
r2_0 = np.array([-10, 30], dtype = float)
k2 = np.array([1, -2], dtype = float)
sigma1 = .1
sigma2 = .1
w1 = np.linalg.norm(k1)
w2 = np.linalg.norm(k2)
t_list = np.linspace(-8, 28, 30)

print(folder_name)
output_folder_name = os.path.dirname(folder_name) + "/free_electrons"
try: os.makedirs(output_folder_name, exist_ok = True)
except: print("Error creating the output folder!")
i = 0
t = t_list[i]
t = 0
r1 = r1_0 + k1 * t
r2 = r2_0 + k2 * t
Z1 = np.array([[np.exp(1j * (k1[0] * x + k1[1] * y) - 1j * w1 * t) * np.exp(-sigma1 * ((x - r1[0]) ** 2 + (y - r1[1]) ** 2)) for x in x_values] for y in y_values], dtype = complex)
Z2 = np.array([[np.exp(1j * (k2[0] * x + k2[1] * y) - 1j * w2 * t) * np.exp(-sigma2 * ((x - r2[0]) ** 2 + (y - r2[1]) ** 2)) for x in x_values] for y in y_values], dtype = complex)
Z_img = np.clip(orbital_render(Z1, upsample = 1), a_min = 0, a_max = 1)
#fig = plt.gcf()
#fig.set_size_inches((7, 7))
frame_number = f"/fframe_{i:0{2}d}.png"
plt.imsave(output_folder_name + frame_number, Z_img)
#plt.show()
#plt.imshow(Z_img)

Band unfolding (work in progress)

In [ ]:
N_ribbon = 4
len_sc = 8
dopant = "N"
dopant_row = 2

folder_name_pc = "C:/DFT/" + str(N_ribbon) + "-AGNR/" + str(N_ribbon) + "-AGNR_primitive/"
file_name_pc = folder_name_pc + "CONTCAR"

structure_pc, lattice_pc, center_pc = structure_loader(file_name_pc)
n_bands_pc, n_kpoints_pc, n_ions, n_noH, n_orbitals, spin_polarized, orbital_list, pz_index, luco_index_pc, hoco_energy_pc, luco_energy_pc, midgap_energy_pc, band_gap_pc, procar_data_pc = procar_loader(folder_name_pc)
pc_latt = lattice_pc.matrix

folder_name_sc = "C:/DFT/" + str(N_ribbon) + "-AGNR/" + str(N_ribbon) + "-AGNR_" + str(len_sc) + "_" + dopant + "_row" + str(dopant_row) + "/"
file_name_sc = folder_name_sc + "CONTCAR"

structure_sc, lattice_sc, center_sc = structure_loader(file_name_sc)
n_bands_sc, n_kpoints_sc, n_ions, n_noH, n_orbitals, spin_polarized, orbital_list, pz_index, luco_index_sc, hoco_energy_sc, luco_energy_sc, midgap_energy_sc, band_gap_sc, procar_data_sc = procar_loader(folder_name_sc)
sc_latt = lattice_sc.matrix

sc_matrix = np.diag([lattice_sc.a / lattice_pc.a, lattice_sc.b / lattice_pc.b, lattice_sc.c / lattice_pc.c])
print(sc_matrix)

In [ ]:
atoms_pc = AseAtomsAdaptor.get_atoms(structure_pc)
pc_opts = unfold.get_symmetry_dataset(atoms_pc)
atoms_sc = AseAtomsAdaptor.get_atoms(structure_sc)
sc_opts = unfold.get_symmetry_dataset(atoms_sc)

unfold_obj = unfold.Unfold(sc_matrix, folder_name_pc + "WAVECAR")

In [394]:
unfold_obj = unfold.UnfoldKSet(sc_matrix, k_points_pc, pc_latt, pc_opts.rotations, sc_opts.rotations)

In [30]:
from vaspwfc import vaspwfc
from unfold import removeDuplicateKpoints, find_K_from_k, save2VaspKPOINTS, unfold

In [ ]:
kpath = procar_data_pc.kpoints

K_in_sup = []
for kk in kpath:
    kg, g = find_K_from_k(kk, sc_matrix)
    K_in_sup.append(kg)

print(np.array(K_in_sup))

reducedK, kid = removeDuplicateKpoints(K_in_sup, return_map = True)
reducedK
save2VaspKPOINTS(reducedK)

In [ ]:
wavesuper = unfold(M = sc_matrix, wavecar = folder_name_sc + "WAVECAR")

from unfold import EBS_cmaps
sw = wavesuper.spectral_weight(K_in_sup)
e0, sf = wavesuper.spectral_function(nedos = 4000)
# or show the effective band structure with colormap
EBS_cmaps(kpath, pc_latt, e0, sf, nseg = 39, eref = -1.01, show = False, ylim = (-7, 7))

Tight-binding

In [ ]:
N_GNR = 5
orientation = "A"
structure = build_GNR(N = N_GNR, orientation = orientation)
fig = structure_plot(structure, supercell = (3, 1, 1), dimensions = 2, model_type = "TB", bond_size = .4, return_type = "figure")
#fig.savefig(folder_name + "TBmodel.svg")
fig.show()

In [ ]:
H_nn = find_neighbors(structure, max_bondlength = 1.6, exclude_hydrogen = True)
H_2n = find_neighbors(structure, min_bondlength = 1.6, max_bondlength = 2.6, exclude_hydrogen = True)
H_3n = find_neighbors(structure, min_bondlength = 2.6, max_bondlength = 2.9, exclude_hydrogen = True)

# Prepare symbols gamma = exp(i k.a) and conjugate gamma
gam, con_gam = symbols("g g*")

# Prepare perturbation Hamiltonians
delta_epsilon = .8
delta_t = .8
H_pert_row2 = np.zeros_like(H_nn, dtype = float)
H_pert_row3 = np.zeros_like(H_nn, dtype = float)
# Perturbation Hamiltonians corresponding to localized defects
H_pert_row2[1, 1] = delta_epsilon # The on-site energy is shifted
H_pert_row3[2, 2] = delta_epsilon # The on-site energy is shifted

# Prepare the projection operator matrix
P = np.zeros_like(H_nn, dtype = float)

# Coupling Hamiltonians
Hk_nn = Matrix(np.zeros_like(H_nn))
Hk_2n = Matrix(np.zeros_like(H_nn))
Hk_3n = Matrix(np.zeros_like(H_nn))

if N_GNR == 5:
    Hk_nn[1, 6], Hk_nn[3, 8] = [gam, gam]
    Hk_nn[6, 1], Hk_nn[8, 3] = [con_gam, con_gam]

    Hk_2n[1, 0], Hk_2n[1, 2], Hk_2n[3, 2], Hk_2n[3, 4] = [gam, gam, gam, gam]
    Hk_2n[0, 1], Hk_2n[2, 1], Hk_2n[2, 3], Hk_2n[4, 3] = [con_gam, con_gam, con_gam, con_gam]

    Hk_2n[6, 5], Hk_2n[6, 7], Hk_2n[8, 7], Hk_2n[8, 9] = [con_gam, con_gam, con_gam, con_gam]
    Hk_2n[5, 6], Hk_2n[7, 6], Hk_2n[7, 8], Hk_2n[9, 8] = [gam, gam, gam, gam]

    Hk_3n[0, 5], Hk_3n[1, 8], Hk_3n[2, 7], Hk_3n[3, 6], Hk_3n[4, 9] = [con_gam, gam, con_gam, gam, con_gam]
    Hk_3n[5, 0], Hk_3n[8, 1], Hk_3n[7, 2], Hk_3n[6, 3], Hk_3n[9, 4] = [gam, con_gam, gam, con_gam, gam]

if N_GNR == 5:
    P[0] = [-1, 0, 1, 0, -1, -1, 0, 1, 0, -1] # B1u, flat band
    P[1] = [-1, 0, 1, 0, -1, 1, 0, -1, 0, 1] # B2g, flat band

    P[2] = [.5, .5 * sqrt(3), 1., .5 * sqrt(3), .5, .5, .5 * sqrt(3), 1., .5 * sqrt(3), .5] # B1u, gamma-type band
    P[3] = [-.5, .5 * sqrt(3), -1., .5 * sqrt(3), -.5, .5, -.5 * sqrt(3), 1., -.5 * sqrt(3), .5] # B2g, gamma-type band
    P[4] = [.5, -.5 * sqrt(3), 1., -.5 * sqrt(3), .5, .5, -.5 * sqrt(3), 1., -.5 * sqrt(3), .5] # B3g, gamma-type band
    P[5] = [-.5, -.5 * sqrt(3), -1., -.5 * sqrt(3), -.5, .5, .5 * sqrt(3), 1., .5 * sqrt(3), .5] # Au, gamma-type band

    P[6] = [1, 0, 0, 0, -1, 1, 0, 0, 0, -1] # B3g, SSH band
    P[7] = [0, 1, 0, -1, 0, 0, 1, 0, -1, 0] # B3g, SSH band
    P[8] = [-1, 0, 0, 0, 1, 1, 0, 0, 0, -1] # Au, SSH band
    P[9] = [0, -1, 0, 1, 0, 0, 1, 0, -1, 0] # Au, SSH band

# Normalize the projection operator matrix, so that it becomes unitary
for i in range(len(P)): P[i] /= np.sqrt(np.sum(P[i] ** 2))
#plt.imshow([[sum(P[i] * P[j]) for i in range(10)] for j in range(10)]) # Check orthonormality
#plt.show()

# Perform the change of basis to all the individual matrices
H_nn_sym = P @ H_nn @ np.linalg.inv(P)
Hk_nn_sym = P @ Hk_nn @ np.linalg.inv(P)
H_2n_sym = P @ H_2n @ np.linalg.inv(P)
Hk_2n_sym = P @ Hk_2n @ np.linalg.inv(P)
H_3n_sym = P @ H_3n @ np.linalg.inv(P)
Hk_3n_sym = P @ Hk_3n @ np.linalg.inv(P)

H_pert_sym_row2 = P @ H_pert_row2 @ np.linalg.inv(P)
H_pert_sym_row3 = P @ H_pert_row3 @ np.linalg.inv(P)

k = .5 * pi

atomistic_Hamiltonians = np.hstack([1.5 * H_nn + np.imag(np.array(Hk_nn.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex)), 0 * H_nn[0:2].T,
                                    1.5 * H_2n + np.imag(np.array(Hk_2n.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex)), 0 * H_nn[0:2].T,
                                    1.5 * H_3n + np.imag(np.array(Hk_3n.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex))])
symmetrized_Hamiltonians = np.hstack([1.5 * H_nn_sym + np.imag(np.array(Hk_nn_sym.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex)), 0 * H_nn[0:2].T,
                                      1.5 * H_2n_sym + np.imag(np.array(Hk_2n_sym.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex)), 0 * H_nn[0:2].T,
                                      1.5 * H_3n_sym + np.imag(np.array(Hk_3n_sym.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex))])

Hamiltonians_pert_row2 = np.hstack([1.5 * (H_nn + H_pert_row2) + np.imag(np.array(Hk_nn.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex)), 0 * H_nn[0:2].T,
                                    1.5 * H_pert_sym_row2, 0 * H_nn[0:2].T,
                                    1.5 * (H_nn_sym + H_pert_sym_row2) + np.imag(np.array(Hk_nn_sym.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex))])
Hamiltonians_pert_row3 = np.hstack([1.5 * (H_nn + H_pert_row3) + np.imag(np.array(Hk_nn.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex)), 0 * H_nn[0:2].T,
                                    1.5 * H_pert_sym_row3, 0 * H_nn[0:2].T,
                                    1.5 * (H_nn_sym + H_pert_sym_row3) + np.imag(np.array(Hk_nn_sym.subs({gam: exp(1j * k), con_gam: exp(-1j * k)}), dtype = complex))])

fig = grid_plot(np.hstack([atomistic_Hamiltonians, 0 * H_nn[0:4].T, symmetrized_Hamiltonians]), return_type = "figure")
fig.show()

fig = grid_plot(Hamiltonians_pert_row2, return_type = "figure")
fig.show()
#fig.savefig(folder_name + "/perturbed_Hamiltonians_row2_grid.svg")
fig = grid_plot(Hamiltonians_pert_row3, return_type = "figure")
fig.show()
#fig.savefig(folder_name + "/perturbed_Hamiltonians_row3_grid.svg")

In [ ]:
k_points = np.linspace(-pi, pi, 41)
all_eigs = []
all_eigvs = []

t1 = -2.7
t2 = 0
t3 = 0

# Get the band structure by diagonalizing at all the k-points
for k in k_points:
    Hk_nn_sym_num = np.array(Hk_nn_sym.subs({gam: np.exp(1j * k), con_gam: np.exp(-1j * k)}), dtype = complex) + H_nn_sym
    Hk_2n_sym_num = np.array(Hk_2n_sym.subs({gam: np.exp(1j * k), con_gam: np.exp(-1j * k)}), dtype = complex) + H_2n_sym
    Hk_3n_sym_num = np.array(Hk_2n_sym.subs({gam: np.exp(1j * k), con_gam: np.exp(-1j * k)}), dtype = complex) + H_3n_sym

    Hk_nn_num = np.array(Hk_nn.subs({gam: np.exp(1j * k), con_gam: np.exp(-1j * k)}), dtype = complex) + H_nn
    
    H_upper = (t1 * Hk_nn_sym_num + t2 * Hk_2n_sym_num + t3 * Hk_3n_sym_num)[:6, :6]
    H_lower = (t1 * Hk_nn_sym_num + t2 * Hk_2n_sym_num + t3 * Hk_3n_sym_num)[6:10, 6:10]

    #eigenvalues, eigenvectors_unordered = np.linalg.eig(t1 * Hk_nn_sym_num + t2 * Hk_2n_sym_num + t3 * Hk_3n_sym_num)
    eigenvalues, eigenvectors_unordered = np.linalg.eig(H_lower)
    eigenvectors = eigenvectors_unordered
    eigenvalues = np.real(eigenvalues)
    ordering = np.argsort(eigenvalues)
    eigenvectors = eigenvectors_unordered[:, ordering]
    eigenvalues = [eigenvalues[ordering[i]] for i in range(len(ordering))] #np.sort(np.real(eigenvalues))
    all_eigs.append(eigenvalues)
    all_eigvs.append(eigenvectors)
all_eigs = np.array(all_eigs)
all_eigvs = np.array(all_eigvs)

[plt.plot(k_points, all_eigs[:, i]) for i in range(len(all_eigs[0]))]
plt.gca().set_aspect(1)
plt.show()

In [ ]:
# Tool to plot orbitals or SALCs

k_point = 0
fig, axs = plt.subplots(1, 10)

for orb_no in range(10):
    #orb = 3 * np.abs(all_eigvs[k_point, :, orb_no]) ** 2
    #orbsgn = np.sign(np.real(all_eigvs[k_point, :, orb_no]))

    orb = 3 * np.abs(P[orb_no]) ** 2
    orbsgn = np.sign(np.real(P[orb_no]))
    pc = PatchCollection([plt.Circle(structure.cart_coords[i, :2], orb[i], color = (.5 * orbsgn[i] + .5, 0, .5 - .5 * orbsgn[i], .8)) for i in range(len(orb))], match_original = True)

    structure_ext = extend_structure(structure)[2]

    atoms, bonds = create_2D_model(structure_ext, atom_size = .1, bond_size = .4, max_bondlength = 1.6, opacity = 1, contrast = .5, brightness = .75)
    bonds_collection = PatchCollection(bonds, match_original = True)
    atoms_collection = PatchCollection(atoms, match_original = True)
    
    axs[orb_no].add_collection(bonds_collection)
    axs[orb_no].add_collection(atoms_collection)

    axs[orb_no].add_collection(pc)
    axs[orb_no].set_xlim([0, structure.lattice.a])
    axs[orb_no].set_ylim([0, structure.lattice.b])
    axs[orb_no].set_aspect(1)
fig.set_size_inches(16, 6)
plt.tight_layout()
#plt.savefig(folder_name + "/SALC_basis.svg")
plt.show()

Graphene and GNRs - band structures from NN TB and cutting

In [ ]:
lim = 1.1547 * pi
kx_list = np.linspace(-lim, lim, 101)
ky_list = np.linspace(-lim, lim, 101)
X, Y = np.meshgrid(kx_list, ky_list)

Z1 = np.array([[np.abs(2 * cos(.5 * sqrt(3) * ky) * np.exp(-.5j * kx) + np.exp(1j * kx)) for kx in kx_list] for ky in ky_list])
Z2 = np.array([[-np.abs(2 * cos(.5 * sqrt(3) * ky) * np.exp(-.5j * kx) + np.exp(1j * kx)) for kx in kx_list] for ky in ky_list])
Z1 = np.array([[kx ** 2 + ky ** 2 for kx in kx_list] for ky in ky_list])
fig, ax = plt.subplots(subplot_kw = {"projection": "3d"})
#plt.pcolormesh(X, Y, Z)
#plt.figaspect(.1)
ax.plot_surface(X, Y, Z1, cmap = "viridis")
#ax.plot_surface(X, Y, Z2, cmap = "viridis")
ax.set_proj_type("ortho")
ax.set_box_aspect([1, 1, .8])
ax.set_xlabel("kx")
ax.set_ylabel("ky")
ax.set_zlabel("energy")
plt.tight_layout()
#plt.savefig(folder_name + "/free_electron_disp_3D.svg")
plt.show()

In [ ]:
def TB_band_structure(kx, ky): return np.abs(2 * cos(.5 * np.sqrt(3) * ky) * exp(-.5j * kx) + exp(1j * kx))
kx_list = np.linspace(-pi / 3, pi / 3, 101)
fig, ax = plt.subplots(1, 9)
for i in range(len(ax)):
    N = i + 4
    ky_list = np.linspace(2 * pi / (sqrt(3) * (N + 1)), 2 * N * pi / (sqrt(3) * (N + 1)), N)
    [ax[i].plot(kx_list, [-2.7 * TB_band_structure(kx, ky_list[ny]) for kx in kx_list], c = (mod(ny, 2) * .7, 0, (1 - mod(ny, 2)) * .8)) for ny in range(len(ky_list))]
    [ax[i].plot(kx_list, [2.7 * TB_band_structure(kx, ky_list[ny]) for kx in kx_list], c = (mod(ny, 2) * .7, 0, (1 - mod(ny, 2)) * .8)) for ny in range(len(ky_list))]
    ax[i].set_xlabel("kx")
    ax[i].set_xticks([-2, 2])
    ax[i].set_xlim([-pi / 3, pi / 3])
    ax[i].set_ylim([-8, 8])
    ax[i].set_title("N = " + str(N) + "-AGNR")
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, wspace=0.4, hspace=0.4)
fig.set_size_inches(18, 5)
plt.savefig("C:\\Presentations\\AGNR_NN-TB_bandstructures.svg")
plt.show()